# Employee Salary Hike Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `salary_hike_pct`

## 2 · Project Overview

We predict **employee salary hike percentage** based on performance rating, tenure, current salary, department, education, project count, training hours, and promotion history.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given an employee's performance rating, tenure, current compensation, department, education level, and work history, predict their annual salary hike percentage.

## 5 · Why This Project Matters

- **Compensation planning** is critical for HR budgeting and retention.
- Understanding hike drivers enables fair and transparent pay decisions.
- Predicting hikes helps employees set expectations.
- Demonstrates regression with HR/people analytics features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | years_at_company, years_experience, performance_rating, current_salary, department, education, num_projects, training_hours, last_promotion_years, is_remote |
| **Target** | salary_hike_pct (continuous, %) |
| **Range** | ~0% – 25% |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "salary_hike_pct"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 employee records with performance, tenure, and compensation features.

In [ ]:
np.random.seed(SEED)
n = 7000
years_at_company = np.random.randint(0, 25, n)
years_experience = (years_at_company + np.random.randint(0, 10, n)).clip(0, 35)
performance_rating = np.round(np.random.normal(3.5, 0.8, n).clip(1, 5), 1)
current_salary = np.round(30000 + 2500 * years_experience + 5000 * performance_rating
                          + np.random.normal(0, 8000, n), 0).clip(25000, 250000).astype(int)
department = np.random.choice(["Engineering", "Sales", "Marketing", "HR", "Finance", "Operations"], n,
                               p=[0.25, 0.2, 0.15, 0.1, 0.15, 0.15])
dept_mult = {"Engineering": 1.15, "Sales": 1.1, "Marketing": 1.0, "HR": 0.95,
             "Finance": 1.05, "Operations": 0.95}
dept_val = np.array([dept_mult[d] for d in department])

education = np.random.choice(["High School", "Bachelor", "Master", "PhD"], n,
                              p=[0.1, 0.45, 0.35, 0.1])
edu_bonus = {"High School": 0, "Bachelor": 0.5, "Master": 1.0, "PhD": 1.5}
edu_val = np.array([edu_bonus[e] for e in education])

num_projects = np.random.poisson(5, n).clip(0, 20)
training_hours = np.round(np.random.exponential(20, n).clip(0, 120), 0).astype(int)
last_promotion_years = np.random.randint(0, 8, n)
is_remote = np.random.binomial(1, 0.35, n)

# Salary hike model (percentage)
base_hike = 3.0
hike = (base_hike
        + 1.5 * (performance_rating - 3)
        + edu_val
        + 0.1 * num_projects
        + 0.02 * training_hours
        - 0.15 * years_at_company
        + 0.5 * last_promotion_years
        - 0.00001 * current_salary)
hike = hike * dept_val
salary_hike_pct = np.round(hike + np.random.normal(0, 1.2, n), 2).clip(0, 25)

df = pd.DataFrame({
    "years_at_company": years_at_company, "years_experience": years_experience,
    "performance_rating": performance_rating, "current_salary": current_salary,
    "department": department, "education": education, "num_projects": num_projects,
    "training_hours": training_hours, "last_promotion_years": last_promotion_years,
    "is_remote": is_remote, "salary_hike_pct": salary_hike_pct
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['salary_hike_pct'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["performance_rating"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Performance Rating"); axes[0][0].set_ylabel("Hike %")
axes[0][0].set_title("Performance vs Hike")

axes[0][1].scatter(df["years_at_company"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Years at Company"); axes[0][1].set_ylabel("Hike %")
axes[0][1].set_title("Tenure vs Hike")

axes[0][2].scatter(df["current_salary"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("Current Salary"); axes[0][2].set_ylabel("Hike %")
axes[0][2].set_title("Salary vs Hike")

df.boxplot(column=TARGET, by="department", ax=axes[1][0])
axes[1][0].set_title("Hike by Department")
axes[1][0].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="education", ax=axes[1][1])
axes[1][1].set_title("Hike by Education")
axes[1][1].tick_params(axis="x", rotation=45)

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `salary_hike_pct`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Salary Hike (%)")

df.boxplot(column=TARGET, by="department", ax=axes[1])
axes[1].set_title("Hike by Department")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print(f"Range: {df[TARGET].min():.1f}% to {df[TARGET].max():.1f}%")
print(f"Mean: {df[TARGET].mean():.2f}%, Median: {df[TARGET].median():.2f}%")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `department`, `education` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Very high hikes (>15%) are genuine top-performer rewards.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["tenure_ratio"] = X_train["years_at_company"] / (X_train["years_experience"] + 1)
X_test["tenure_ratio"] = X_test["years_at_company"] / (X_test["years_experience"] + 1)

X_train["performance_x_projects"] = X_train["performance_rating"] * X_train["num_projects"]
X_test["performance_x_projects"] = X_test["performance_rating"] * X_test["num_projects"]

X_train["promotion_gap"] = X_train["last_promotion_years"] * X_train["performance_rating"]
X_test["promotion_gap"] = X_test["last_promotion_years"] * X_test["performance_rating"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Performance rating** is the strongest hike predictor (~1.5% per point above 3.0).
- **Engineering and Sales** departments get higher hikes (~15% and 10% premium).
- **Education level** adds incremental hike (PhD: +1.5% vs High School).
- **Tenure** slightly reduces hike (diminishing marginal returns).
- **Last promotion gap** increases hike (overdue employees get catch-up raises).

**Business takeaway:** Performance drives hikes most — invest in clear performance metrics. Monitor promotion gaps to prevent retention risk.

## 26 · Limitations

1. Synthetic data with simplified compensation dynamics.
2. No market salary benchmarking.
3. Missing company financial performance.
4. No negotiation or offer-matching factors.
5. Simplified department multipliers.

## 27 · How to Improve This Project

1. Use real anonymized HR compensation data.
2. Add market salary benchmarks (Glassdoor, levels.fyi).
3. Include company revenue growth and budget constraints.
4. Model equity/stock compensation separately.
5. Add peer comparison features.

## 28 · Production Considerations

- Deploy for HR compensation planning dashboards.
- Use for hike budget allocation by department.
- Monitor for pay equity across demographics.
- Update annually with market data.
- Output prediction intervals for negotiation ranges.

## 29 · Common Mistakes

1. Not accounting for current salary level (high earners get lower %).
2. Ignoring department-specific hike budgets.
3. Treating tenure as always positive (diminishing returns).
4. Not considering promotion history gaps.
5. Using raw years instead of tenure-to-experience ratio.

## 30 · Mini Challenge / Exercises

1. Remove `performance_rating` — how much does R² drop?
2. Create `salary_band = pd.qcut(current_salary, 4)` and analyze hikes per band.
3. Plot hike vs. last_promotion_years — confirm the catch-up effect.
4. Build separate models by department.
5. Check if `is_remote` has any predictive power.

## 31 · Final Summary / Key Takeaways

1. **Performance rating** is the dominant hike driver.
2. **Department** sets the hike budget tier.
3. **Education level** adds incremental hike percentage.
4. **Tenure** has diminishing returns on hike size.
5. **Promotion gaps** trigger catch-up raises.
6. **Fair compensation** requires transparent, data-driven hike models.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))